# Indexing NTCIR-2 AdHoc by OpenSearch for BM25 Model

Prerequisite
  - [ntcir2-adhoc_preprocess.ipynb](../../dataset/ntcir2-adhoc/ntcir2-adhoc-preprocess.ipynb)

In [1]:
import sys
!{sys.executable} -m pip install -q ir_datasets pandas opensearch-py dotenv

In [2]:
import pprint
from tqdm import tqdm

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [3]:

import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'qIJ28Ej2TCC6_LI88zZLlw',
 'name': '4db878c40bab',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2026-02-07T07:54:31.169913465Z',
             'build_hash': 'bbc94f0bdc3a759011e6529ecfe52840856f91a3',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.2',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.5.0'}}


Install opensearch plugins for Japanese processing
  - Requires a restart of docker container after the installation

In [ ]:
!docker exec -it opensearch-node \
  /usr/share/opensearch/bin/opensearch-plugin install --batch analysis-kuromoji analysis-icu
!docker restart opensearch-node

In [4]:
import os
sys.path.append(os.path.join(os.getcwd(), '..', '..', 'dataset', 'ntcir2-adhoc'))

In [5]:
import ir_datasets
import ntcir2_adhoc
dataset = ir_datasets.load('ntcir2-adhoc')

In [6]:
index_name = "ntcir2_bm25"

In [7]:
# Delete an existing index (be careful)
if client.indices.exists(index=index_name):
    response = client.indices.delete(index=index_name)
    pprint.pprint(response)
else:
    print(f"{index_name} does not exist")

ntcir2_bm25 does not exist


In [8]:

index_body = {
  "settings": {
    "index": {
      "number_of_shards": 1
    },
    "analysis":{
      "analyzer": {
        "default": {
            "type": "custom",
            "tokenizer": "kuromoji_tokenizer",
            "char_filter": ["icu_normalizer"],
            "filter": [
                "kuromoji_baseform",
                "kuromoji_stemmer",
                "kuromoji_part_of_speech",
                "cjk_width",
                "ja_stop"
            ]
        }
      }
    }
  },
  "mappings": {
    "properties": {
        "docid": { "type": "keyword" },
        "title": { "type": "text" },
        "text": { "type": "text" },
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)
pprint.pprint(response)

{'acknowledged': True, 'index': 'ntcir2_bm25', 'shards_acknowledged': True}


In [9]:
def prepare_documents(dataset):
    """
    Prepare individual documents for indexing, with progress tracking.
    """
    docstore = dataset.docs_store()
    total_docs = sum(1 for _ in dataset.docs_iter())  # Count the total documents
    progress = tqdm(total=total_docs, desc="Indexing Documents")  # Progress bar

    for doc in dataset.docs_iter():
        # Parse the document
        title, text = docstore.get(doc.doc_id).text.split(None, 1)
        text = text.replace("\n", " ")
        yield {
            "_id": doc.doc_id,  # Unique identifier for the document
            "_source": {
                "docid": doc.doc_id,
                "text": text,
                "title": title
            }
        }
        progress.update(1)  # Update progress bar

    progress.close()  # Close the progress bar

In [10]:
from opensearchpy.helpers import bulk
success, failed = bulk(client, prepare_documents(dataset), index=index_name)

Indexing Documents: 100%|██████████| 736158/736158 [03:36<00:00, 3398.81it/s]


In [11]:
# Avoid accidental delete
response = client.cluster.put_settings(
    body={
        "persistent": {
            "action.destructive_requires_name": True
        }
    }
)
pprint.pprint(response)

{'acknowledged': True,
 'persistent': {'action': {'destructive_requires_name': 'true'}},
 'transient': {}}


#### Search Test

In [12]:
def search(query: str, size: int = 10) -> dict:
    body = {
        "size": size,
        "query": {
            "multi_match": {
                "query": query,
                "fields": ["title^2", "text"] # title gets a boost
            }
        },
    }

    return client.search(index=index_name, body=body)

In [ ]:
q = "情報検索システム"
resp = search(q, size=5)

print(f"\nTop {len(resp['hits']['hits'])} hits for query: {q}\n")
for hit in resp["hits"]["hits"]:
    src = hit["_source"]
    print(f"[{src['docid']}] {src['title'][:50]}... (score={hit['_score']:.2f})")